# GCN+BiGRU NIDS: Context-Window Temporal–Spatial Evaluation Pipeline

This notebook implements a **Network Intrusion Detection System (NIDS)** using a
**GCN + BiGRU** context-window pipeline for **per-flow intrusion detection**.

**Supported datasets:** NSL-KDD, CICIDS-2017, UAVIDS-2025

**Model design:**
- *Temporal axis*: sliding window of T=20 consecutive flows (sorted by timestamp
  when available, otherwise by row order). NSL-KDD uses context-window row ordering
  due to lack of true timestamps.
- *Spatial axis*: fully-connected graph built among the 20 flows within each window (configurable via `GRAPH_MODE`; set to `"knn"` for sparse kNN graph).
- *GCN*: `GCNConv` layers perform spatial message passing over the within-window graph.
- *BiGRU*: processes the ordered sequence of T node embeddings; the last-step output
  is used for classification.
- *Label*: predict the class of the **last flow** in each window (per-flow labeling).

**Execution order:**
1. Run the *PyG Installation* cell (if needed).
2. Run *Imports* and *Setup* cells in order.
3. Run *Step 1* to launch the full evaluation pipeline.


# PyG installation — run once if torch_geometric is not yet installed.
# Installs wheels that match the currently installed PyTorch build.
import sys
import torch

torch_base = torch.__version__.split('+')[0]
torch_build = torch.__version__.split('+')[1] if '+' in torch.__version__ else 'cpu'
pyg_wheel_url = f"https://data.pyg.org/whl/torch-{torch_base}+{torch_build}.html"
print(f"Using PyG wheel index: {pyg_wheel_url}")

!{sys.executable} -m pip install --upgrade --force-reinstall --no-cache-dir torch_scatter torch_sparse torch_cluster torch_spline_conv -f {pyg_wheel_url}
!{sys.executable} -m pip install --upgrade torch-geometric

In [ ]:
# Standard library
import copy, gc, importlib, math, os, random, sys, time, tracemalloc
from itertools import combinations

# Visualisation
import matplotlib.pyplot as plt

# Data science
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Scikit-learn
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import (
    LabelBinarizer,
    LabelEncoder,
    OrdinalEncoder,
    StandardScaler,
)

# PyTorch Geometric (requires installation — see cell above)
from torch_geometric.data import Batch as PyGBatch
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GCNConv


In [ ]:
# Sanity-check: verify PyG and supporting packages are installed.
print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')

for pkg in ['torch_geometric', 'torch_scatter', 'torch_sparse', 'torch_cluster']:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'installed')
        print(f'{pkg:<20}: {ver}')
    except Exception as exc:
        print(f"{pkg:<20}: FAILED TO LOAD ({type(exc).__name__}: {exc})")
        print('  Re-run Cell 2 to reinstall matching wheels for this torch build.')

In [ ]:
# Device configuration — options: 'auto', 'cpu', 'cuda'
DEVICE_MODE = 'auto'

if DEVICE_MODE == 'cpu':
    device = torch.device('cpu')
elif DEVICE_MODE == 'cuda':
    if not torch.cuda.is_available():
        raise RuntimeError("DEVICE_MODE='cuda' but CUDA is not available.")
    device = torch.device('cuda')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Selected device mode : {DEVICE_MODE}')
print(f'Using device         : {device}')

In [ ]:
def _downcast_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns to float32/int32 to halve CPU RAM usage."""
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    return df


# NSL-KDD column names (no header in the raw files)
nsl_columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land',
    'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'attack', 'level',
]

nsl_train = _downcast_dataframe(pd.read_csv('./dataset/nslkdd/KDDTrain+.txt', header=None))
nsl_test = _downcast_dataframe(pd.read_csv('./dataset/nslkdd/KDDTest+.txt', header=None))
nsl_train.columns = nsl_columns
nsl_test.columns = nsl_columns
nsl_kdd_fresh = pd.concat([nsl_train, nsl_test], ignore_index=True)

cicd2017 = _downcast_dataframe(
    pd.read_csv('./dataset/cicids/Wednesday-workingHours.pcap_ISCX.csv')
)
uavids = _downcast_dataframe(pd.read_csv('./dataset/uavids/UAVIDS-2025.csv'))

# Shared registry used by all pipeline stages
datasets = {
    'CICIDS2017': cicd2017,
    'NSL-KDD': nsl_kdd_fresh,
    'UAVIDS': uavids,
}

target_columns = {
    'NSL-KDD': 'attack',
    'CICIDS2017': ' Label',
    'UAVIDS': 'label',
}

print('Loaded datasets:', {k: v.shape for k, v in datasets.items()})

## Utility Functions

In [ ]:
def set_seed(seed: int = 42) -> None:
    """Set random seed across all libraries for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def preprocess_data(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Clean inf/NaN values while keeping all columns (including target).

    Rows are dropped based on inf/NaN in numeric columns only; the original
    index is reset so callers can reindex target labels safely.
    """
    df = dataframe.copy()
    numeric_cols = df.select_dtypes(include=['number']).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=numeric_cols)
    return df


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Build a compact ColumnTransformer to avoid one-hot memory explosion.

    Numeric columns are standardized; categorical columns are ordinal-encoded.
    Fit on training data and apply transform() on val/test to avoid leakage.
    """
    numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=['number']).columns.tolist()

    transformers = [('num', StandardScaler(), numeric_cols)]
    if categorical_cols:
        transformers.append(
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
        )

    return ColumnTransformer(transformers=transformers, remainder='drop')

## Model Architecture

In [ ]:
class GCN_BiGRU(torch.nn.Module):
    """GCN + BiGRU model for context-window per-flow intrusion detection.

    For each window of T consecutive flows:
      1. GCNConv layers perform spatial message passing over the within-window
         within-window graph (kNN or fully-connected, see ``GRAPH_MODE``),
         producing per-flow embeddings.
      2. A BiGRU processes the ordered sequence of T embeddings to capture
         temporal context.
      3. A classifier head maps the last BiGRU hidden state to class logits,
         predicting the label of the **last flow** in the window.

    Parameters
    ----------
    input_dim : int
        Number of input features per flow.
    hidden_dim : int
        Hidden size for GCN layers and the classifier MLP.
    output_dim : int
        1 for binary (BCEWithLogitsLoss), num_classes for multi-class (CrossEntropyLoss).
    num_gcn_layers : int
        Number of stacked GCNConv layers.
    gru_hidden : int
        Hidden size of each GRU direction.
    num_gru_layers : int
        Number of stacked BiGRU layers.
    dropout : float
        Dropout probability applied after each GCN layer and within the MLP head.
    window_size : int
        Number of flows per context window (T).  Must match the dataset.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        output_dim: int = 1,
        num_gcn_layers: int = 2,
        gru_hidden: int = 64,
        num_gru_layers: int = 1,
        dropout: float = 0.2,
        window_size: int = 20,
    ) -> None:
        super().__init__()
        self.window_size = window_size

        # --- Spatial: GCN layers ---
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        in_ch = input_dim
        for _ in range(num_gcn_layers):
            self.convs.append(GCNConv(in_ch, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
            in_ch = hidden_dim

        self.gcn_dropout = nn.Dropout(p=dropout)

        # --- Temporal: Bidirectional GRU ---
        self.bigru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=gru_hidden,
            num_layers=num_gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_gru_layers > 1 else 0.0,
        )

        # --- Classifier head ---
        self.classifier = nn.Sequential(
            nn.Linear(2 * gru_hidden, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, data: PyGBatch) -> torch.Tensor:
        """Forward pass over a batched PyG graph.

        Parameters
        ----------
        data : PyGBatch
            Batched graph produced by _pyg_collate.  Each graph in the batch
            corresponds to one context window with exactly window_size nodes.

        Returns
        -------
        torch.Tensor
            Logits of shape (B,) (binary) or (B, num_classes) (multi-class).
        """
        x = data.x               # (B*T, F)
        edge_index = data.edge_index  # (2, E) — global node indices across batch
        B = data.num_graphs
        T = self.window_size

        # 1. Spatial: GCN message passing over within-window graphs
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
            x = self.gcn_dropout(x)
        # x: (B*T, H)

        # 2. Reshape to sequence: (B, T, H)
        #    PyG Batch concatenates nodes in window order, so view is safe.
        x = x.view(B, T, -1)

        # 3. Temporal: BiGRU over the window sequence
        gru_out, _ = self.bigru(x)   # (B, T, 2*gru_hidden)
        last = gru_out[:, -1, :]     # (B, 2*gru_hidden) — last-step hidden state

        # 4. Classify
        logits = self.classifier(last)   # (B, output_dim)
        return logits.squeeze(-1) if logits.size(-1) == 1 else logits


## Graph Construction

Supports two modes (controlled by `GRAPH_MODE` in the pipeline configuration cell):

- **`fully_connected`** *(default)*: every flow in the window connects to every other flow — dense spatial mixing within each T=20 context window.
- **`knn`**: each flow connects to its k nearest neighbours in feature space (original behaviour; set `GRAPH_MODE = "knn"` to use).


In [ ]:
def adaptive_graph_construction(
    X: np.ndarray,
    y: np.ndarray,
    k: int = 10,
    metric: str = 'euclidean',
    add_self_loops: bool = True,
) -> Data:
    """Build a kNN graph without an O(n^2) distance matrix.

    Uses PyNNDescent for approximate nearest-neighbour search on large datasets
    (> 1 000 samples), which scales to hundreds of thousands of samples in seconds.
    Falls back to sklearn's kneighbors_graph for small datasets or when pynndescent
    is not installed.

    Parameters
    ----------
    X : node feature matrix, shape (n, d)
    y : node labels, shape (n,)
    k : number of nearest neighbours
    metric : distance metric for kNN
    add_self_loops : if True, add a self-loop for every node

    Returns
    -------
    torch_geometric.data.Data with x, edge_index, y attributes
    """
    n = X.shape[0]
    k_eff = min(k, n - 1)
    _PYNNDESCENT_THRESHOLD = 1000  # use approximate NN for datasets above this size

    rows_list: list = []
    cols_list: list = []

    try:
        if n > _PYNNDESCENT_THRESHOLD:
            from pynndescent import NNDescent
            index = NNDescent(X, metric=metric, n_neighbors=k_eff + 1,
                              random_state=42, verbose=False)
            indices, _ = index.neighbor_graph
            # indices[:, 0] is the node itself — skip it
            indices = indices[:, 1:k_eff + 1]
            for src, neighbors in enumerate(indices):
                for dst in neighbors:
                    rows_list.append(src);      cols_list.append(int(dst))
                    rows_list.append(int(dst));  cols_list.append(src)
        else:
            raise ImportError  # fall through to sklearn for small n
    except (ImportError, Exception):
        adj = kneighbors_graph(X, n_neighbors=k_eff, metric=metric,
                               mode='connectivity', include_self=False)
        # Symmetrise to make the graph undirected
        adj = (adj + adj.T).astype(bool).astype(int)
        adj_coo = adj.tocoo()
        rows_list = adj_coo.row.tolist()
        cols_list = adj_coo.col.tolist()

    rows = np.array(rows_list, dtype=np.int64)
    cols = np.array(cols_list, dtype=np.int64)

    # Remove duplicate edges produced by symmetrisation
    edge_pairs = np.unique(np.vstack([rows, cols]), axis=1)
    rows, cols = edge_pairs[0], edge_pairs[1]

    if add_self_loops:
        self_loop_idx = np.arange(n, dtype=np.int64)
        rows = np.concatenate([rows, self_loop_idx])
        cols = np.concatenate([cols, self_loop_idx])

    edge_index = torch.tensor(np.vstack([rows, cols]), dtype=torch.long)
    return Data(
        x=torch.tensor(X, dtype=torch.float),
        edge_index=edge_index,
        y=torch.tensor(y, dtype=torch.long),
    )

In [ ]:
# ── Window Dataset Builder ────────────────────────────────────────────────

def _build_window_knn_graph(x_win: np.ndarray, k: int) -> torch.Tensor:
    """Build a symmetric kNN graph with self-loops for T flows in a window.

    Parameters
    ----------
    x_win : np.ndarray of shape (T, F)
        Feature matrix for the T flows in the window.
    k : int
        Number of nearest neighbours (clipped to T-1).

    Returns
    -------
    torch.Tensor of shape (2, E)
        ``edge_index`` with 0-indexed node IDs for the window.
    """
    T = len(x_win)
    k_eff = min(k, T - 1)
    self_idx = np.arange(T, dtype=np.int64)

    if k_eff < 1:
        return torch.tensor(np.vstack([self_idx, self_idx]), dtype=torch.long)

    try:
        adj = kneighbors_graph(
            x_win, n_neighbors=k_eff, metric='euclidean',
            mode='connectivity', include_self=False,
        )
        adj = (adj + adj.T).astype(bool).astype(int).tocoo()
        rows = np.array(adj.row, dtype=np.int64)
        cols = np.array(adj.col, dtype=np.int64)
    except Exception:
        # Fallback: no edges beyond self-loops
        rows = np.empty(0, dtype=np.int64)
        cols = np.empty(0, dtype=np.int64)

    rows = np.concatenate([rows, self_idx])
    cols = np.concatenate([cols, self_idx])
    return torch.tensor(np.vstack([rows, cols]), dtype=torch.long)



def _build_window_fully_connected_graph(T: int, add_self_loops: bool = True) -> torch.Tensor:
    """Build a fully-connected undirected graph (both directions) for a window of T nodes.

    Every node connects to every other node in the window, giving the GCN
    complete within-window spatial context.  With T=20 this produces
    20*19 = 380 directed edges plus 20 self-loops per window — small enough
    to keep with any reasonable batch size.

    Parameters
    ----------
    T : int
        Number of nodes (flows) in the window.
    add_self_loops : bool
        When True (default) each node also has a self-loop.

    Returns
    -------
    torch.Tensor of shape (2, E)
        ``edge_index`` with 0-indexed node IDs local to the window.
    """
    if T <= 0:
        return torch.empty((2, 0), dtype=torch.long)
    if T == 1:
        if add_self_loops:
            return torch.tensor([[0], [0]], dtype=torch.long)
        return torch.empty((2, 0), dtype=torch.long)

    pairs = list(combinations(range(T), 2))
    rows = [i for i, j in pairs] + [j for i, j in pairs]
    cols = [j for i, j in pairs] + [i for i, j in pairs]

    if add_self_loops:
        rows += list(range(T))
        cols += list(range(T))

    return torch.tensor([rows, cols], dtype=torch.long)


class FlowWindowDataset(Dataset):
    """Sliding-window dataset for per-flow label prediction.

    Each sample is a context window of ``window_size`` consecutive flows;
    the target is the label of the **last** flow in the window.

    Windows are constructed entirely within one split (train/val/test) to
    prevent any data leakage across splits.

    If timestamp-based ordering is desired, sort ``X`` and ``y`` by time
    **before** passing them to this class.

    Parameters
    ----------
    X : np.ndarray of shape (N, F)
        Pre-processed feature matrix for one split.
    y : array-like of shape (N,)
        Integer labels for each flow.
    window_size : int
        Number of flows per window (T).  Default 20.
    k_spatial : int
        kNN neighbours used when ``graph_mode`` is ``"knn"``.  Clipped to T-1.
    graph_mode : str
        ``"fully_connected"`` (default) — dense within-window graph;
        ``"knn"`` — sparse kNN graph using ``k_spatial`` neighbours.
    """

    def __init__(
        self,
        X: np.ndarray,
        y,
        window_size: int = 20,
        k_spatial: int = 5,
        graph_mode: str = "fully_connected",
    ) -> None:
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.T = window_size
        self.k = min(k_spatial, window_size - 1)
        self.graph_mode = graph_mode
        # Valid end-of-window indices: the first full window ends at index T-1
        self.end_indices = np.arange(window_size - 1, len(self.X))

    def __len__(self) -> int:
        return len(self.end_indices)

    def __getitem__(self, idx: int) -> Data:
        end = self.end_indices[idx]
        x_win = self.X[end - self.T + 1: end + 1]   # (T, F)
        y_last = int(self.y[end])
        if self.graph_mode == "fully_connected":
            edge_index = _build_window_fully_connected_graph(self.T, add_self_loops=True)
        else:
            edge_index = _build_window_knn_graph(x_win, self.k)
        return Data(
            x=torch.tensor(x_win, dtype=torch.float32),
            edge_index=edge_index,
            y=torch.tensor(y_last, dtype=torch.long),
        )


def _pyg_collate(batch):
    """Collate a list of PyG Data objects into a single PyG Batch."""
    return PyGBatch.from_data_list(batch)


## Training and Evaluation

In [ ]:
def _compute_pos_weight(y_tensor):
    """Compute pos_weight = #negatives / #positives for BCEWithLogitsLoss."""
    n_pos = float((y_tensor == 1).sum().item())
    n_neg = float((y_tensor == 0).sum().item())
    return torch.tensor([n_neg / n_pos], dtype=torch.float) if n_pos > 0 else None


def compute_class_weights(y, num_classes: int) -> torch.Tensor:
    """Compute inverse-frequency class weights from training labels.

    Returns a float32 tensor of shape [num_classes] where rarer classes
    receive higher weight, helping the model attend to minority classes.
    """
    counts = pd.Series(y).value_counts().sort_index()
    freq = counts.reindex(range(num_classes), fill_value=0).values.astype(np.float32)
    freq = np.maximum(freq, 1.0)
    weights = freq.sum() / (num_classes * freq)
    return torch.tensor(weights, dtype=torch.float32)


# ── Window-based training helpers ─────────────────────────────────────────

def _evaluate_split_window(model, dataset, num_classes, batch_size, pos_weight=None):
    """Evaluate GCN_BiGRU on a FlowWindowDataset; return per-metric dict."""
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        collate_fn=_pyg_collate, num_workers=0,
    )
    all_logits, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            all_logits.append(logits.detach().cpu())
            all_labels.append(batch.y.detach().cpu())

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)
    labels_np = labels.numpy()

    if num_classes == 2:
        pw = pos_weight if pos_weight is not None else _compute_pos_weight(labels)
        loss_val = F.binary_cross_entropy_with_logits(
            logits.squeeze(), labels.float(),
            pos_weight=pw.cpu() if pw is not None else None,
        ).item()
        preds = (torch.sigmoid(logits).numpy().ravel() > 0.5).astype(int)
    else:
        loss_val = F.cross_entropy(logits, labels.long()).item()
        preds = torch.softmax(logits, dim=1).numpy().argmax(axis=1)

    return {
        'Loss': loss_val,
        'Accuracy': accuracy_score(labels_np, preds),
        'Precision': precision_score(labels_np, preds, average='macro', zero_division=0),
        'Recall': recall_score(labels_np, preds, average='macro', zero_division=0),
        'F1 Score': f1_score(labels_np, preds, average='macro', zero_division=0),
    }


def train_gcn_bigru(
    model,
    train_dataset,
    val_dataset,
    num_classes,
    pos_weight,
    weights_for_loss,
    optimizer,
    scheduler,
    train_epochs,
    patience,
    min_delta,
    grad_clip,
    batch_size,
    print_every: int = 5,
):
    """Train GCN_BiGRU with window DataLoader and early stopping.

    Uses standard ``torch.utils.data.DataLoader`` over ``FlowWindowDataset``
    samples.  Each batch is a PyG ``Batch`` object assembled by ``_pyg_collate``.
    """
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=_pyg_collate, num_workers=0,
    )
    best_val_f1, no_improve_count, best_state = -1.0, 0, None
    train_start = time.perf_counter()

    for epoch in range(train_epochs):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch)
            y = batch.y

            if num_classes == 2:
                loss = F.binary_cross_entropy_with_logits(
                    logits.squeeze(), y.float(), pos_weight=pos_weight
                )
            else:
                loss = F.cross_entropy(logits, y.long(), weight=weights_for_loss)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()

        model.eval()
        val_metrics = _evaluate_split_window(
            model, val_dataset, num_classes, batch_size, pos_weight=pos_weight
        )
        val_f1 = val_metrics['F1 Score']

        if val_f1 > best_val_f1 + min_delta:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            no_improve_count = 0
        else:
            no_improve_count += 1

        if (epoch + 1) % print_every == 0 or epoch == train_epochs - 1:
            print(
                f'  Epoch {epoch + 1:>4d}/{train_epochs}: '
                f'train_loss={epoch_loss:.4f}  val_F1={val_f1:.4f}  '
                f'best_F1={best_val_f1:.4f}'
            )

        if no_improve_count >= patience:
            print(
                f'  Early stopping at epoch {epoch + 1} '
                f'(best val F1={best_val_f1:.4f})'
            )
            break

    train_time = time.perf_counter() - train_start
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, train_time


def evaluate_gcn_bigru(model, test_dataset, num_classes, batch_size):
    """Run test inference for GCN_BiGRU and return a metrics dictionary."""
    loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=_pyg_collate, num_workers=0,
    )
    all_preds, all_probas, all_labels = [], [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            labels = batch.y.cpu().numpy()
            if num_classes == 2:
                proba = torch.sigmoid(logits).cpu().numpy().ravel()
                preds = (proba > 0.5).astype(int)
                all_probas.extend(proba.tolist())
            else:
                proba = torch.softmax(logits, dim=1).cpu().numpy()
                preds = proba.argmax(axis=1)
                all_probas.extend(proba.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_preds)
    y_proba = np.asarray(all_probas)

    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1        = f1_score(y_true, y_pred, average='macro', zero_division=0)
    report    = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    confusion = confusion_matrix(y_true, y_pred)

    auc = None
    try:
        if num_classes == 2:
            auc = roc_auc_score(y_true, y_proba)
        else:
            auc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    except ValueError:
        auc = None

    return {
        'accuracy': accuracy, 'auc': auc, 'precision': precision,
        'recall': recall, 'f1': f1, 'report': report, 'confusion': confusion,
    }


# ── Main benchmark runner ──────────────────────────────────────────────────

def run_all_benchmarks(
    train_dataset,
    val_dataset,
    test_dataset,
    num_classes: int,
    class_weights=None,
    model_hparams=None,
):
    """Train and evaluate the GCN-BiGRU model on window datasets.

    Replaces the previous BG-GCN / NeighborLoader benchmark runner.

    Parameters
    ----------
    train_dataset, val_dataset, test_dataset : FlowWindowDataset
        Window datasets for each split (no leakage guaranteed).
    num_classes : int
        Number of target classes.
    class_weights : torch.Tensor or None
        Inverse-frequency weights for multi-class CE loss.
    model_hparams : dict
        Hyper-parameters (see GCN_BIGRU_PARAMS for keys).

    Returns
    -------
    (results_df, confusion_matrices, split_metrics_tables)
    """
    model_hparams  = model_hparams or {}
    hidden_dim     = int(model_hparams.get('hidden_dim', 64))
    gru_hidden     = int(model_hparams.get('gru_hidden', 64))
    num_gcn_layers = int(model_hparams.get('num_gcn_layers', 2))
    num_gru_layers = int(model_hparams.get('num_gru_layers', 1))
    dropout        = float(model_hparams.get('dropout', 0.2))
    learning_rate  = float(model_hparams.get('lr', 0.001))
    weight_decay   = float(model_hparams.get('weight_decay', 1e-5))
    train_epochs   = int(model_hparams.get('epochs', 50))
    patience       = int(model_hparams.get('patience', 20))
    min_delta      = float(model_hparams.get('min_delta', 1e-4))
    grad_clip      = float(model_hparams.get('grad_clip', 1.0))
    batch_size     = int(model_hparams.get('batch_size', 256))
    window_size    = int(model_hparams.get('window_size', WINDOW_SIZE))

    input_dim  = train_dataset.X.shape[1]
    output_dim = 1 if num_classes == 2 else num_classes

    model = GCN_BiGRU(
        input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim,
        num_gcn_layers=num_gcn_layers, gru_hidden=gru_hidden,
        num_gru_layers=num_gru_layers, dropout=dropout, window_size=window_size,
    ).to(device)

    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    print('\n--- Training GCN-BiGRU ---')
    tracemalloc.start()
    start_time = time.time()

    optimizer = torch.optim.Adam(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, train_epochs)
    )

    # Class-imbalance handling
    weights_for_loss = None
    if class_weights is not None and num_classes > 2:
        w = class_weights.to(device)
        if w.numel() == output_dim:
            weights_for_loss = w

    pos_weight = None
    if num_classes == 2:
        y_train_tensor = torch.tensor(train_dataset.y, dtype=torch.long)
        pw = _compute_pos_weight(y_train_tensor)
        pos_weight = pw.to(device) if pw is not None else None

    model, train_time = train_gcn_bigru(
        model=model,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        num_classes=num_classes,
        pos_weight=pos_weight,
        weights_for_loss=weights_for_loss,
        optimizer=optimizer,
        scheduler=scheduler,
        train_epochs=train_epochs,
        patience=patience,
        min_delta=min_delta,
        grad_clip=grad_clip,
        batch_size=batch_size,
    )

    model.eval()
    test_metrics = evaluate_gcn_bigru(
        model=model, test_dataset=test_dataset,
        num_classes=num_classes, batch_size=batch_size,
    )

    # Per-split metrics table
    split_rows = []
    for split_name, split_ds, split_train_time in [
        ('Trainset',       train_dataset, train_time),
        ('Validation set', val_dataset,   0.0),
        ('Test set',       test_dataset,  0.0),
    ]:
        fwd_start = time.perf_counter()
        m = _evaluate_split_window(
            model, split_ds, num_classes, batch_size,
            pos_weight=pos_weight if num_classes == 2 else None,
        )
        fwd_time = time.perf_counter() - fwd_start
        split_rows.append({
            'Split':                split_name,
            'Train Loop Time (s)':  split_train_time,
            'Forward Time (s)':     fwd_time,
            'Eval Time (s)':        0.0,
            'Computation Time (s)': split_train_time + fwd_time,
            **m,
        })

    split_df = pd.DataFrame(split_rows)[[
        'Split', 'Train Loop Time (s)', 'Forward Time (s)', 'Eval Time (s)',
        'Computation Time (s)', 'Loss', 'Accuracy', 'Precision', 'Recall', 'F1 Score',
    ]]

    end_time = time.time()
    mem_consumption = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

    results = [{
        'Model':                 'GCN-BiGRU',
        'Accuracy':              test_metrics['accuracy'],
        'AUC':                   test_metrics['auc'],
        'Precision':             test_metrics['precision'],
        'Recall':                test_metrics['recall'],
        'F1':                    test_metrics['f1'],
        'Classification Report': test_metrics['report'],
        'Confusion Matrix':      test_metrics['confusion'],
        'Time (s)':              f'{end_time - start_time:.2f} s',
        'Memory (MB)':           f'{mem_consumption / 1e6:.2f} MB',
    }]

    gc.collect()
    torch.cuda.empty_cache()

    results_df = (
        pd.DataFrame(results)
        .drop(columns=['Classification Report', 'Confusion Matrix'])
        .sort_values('Accuracy', ascending=False)
    )
    print('\nBenchmark Results:')
    print(results_df.to_markdown(index=False))

    confusion_matrices   = {'GCN-BiGRU': test_metrics['confusion']}
    split_metrics_tables = {'GCN-BiGRU': split_df}
    return results_df, confusion_matrices, split_metrics_tables


## NSL-KDD Attack Group Mapping

In [ ]:
# Fine-grained NSL-KDD attack labels grouped into five broad categories.
# Defined once here; shared by all experiment stages.
_DOS_ATTACKS = {
    'back', 'land', 'neptune', 'pod', 'smurf', 'teardrop',
    'apache2', 'mailbomb', 'processtable', 'udpstorm',
}
_PROBING_ATTACKS = {'ipsweep', 'nmap', 'portsweep', 'satan', 'mscan', 'saint'}
_U2R_ATTACKS = {'buffer_overflow', 'loadmodule', 'perl', 'rootkit', 'sqlattack', 'xterm', 'ps'}
_R2L_ATTACKS = {
    'ftp_write', 'guess_passwd', 'imap', 'multihop', 'phf', 'spy',
    'warezclient', 'warezmaster', 'httptunnel', 'named', 'sendmail',
    'snmpgetattack', 'snmpguess', 'xlock', 'xsnoop',
}


def _map_nsl_attack_group(label: str) -> str:
    """Map a fine-grained NSL-KDD attack label to one of five broad categories.

    Returns 'normal', 'DoS', 'probing', 'U2R', 'R2L', or NaN for unknown labels.
    """
    label = str(label).strip()
    if label == 'normal':
        return 'normal'
    if label in _PROBING_ATTACKS:
        return 'probing'
    if label in _DOS_ATTACKS:
        return 'DoS'
    if label in _U2R_ATTACKS:
        return 'U2R'
    if label in _R2L_ATTACKS:
        return 'R2L'
    return np.nan

## Step 1: Baseline Evaluation Pipeline
Run the next cell to execute each dataset one-by-one:
1. Experiment with default BG-GCN parameters

Configuration is in the *Pipeline Configuration* cell below:
- `NUM_DATASETS_TO_TEST = 1`, `2`, or `'all'`
- `EXPERIMENT_CFG` for sampling
- `DEVICE_MODE = 'auto' | 'cpu' | 'cuda'` (in the Device Configuration cell)


## GCN + BiGRU Context-Window Design

### Temporal axis (consistent across all datasets)
After preprocessing and train/val/test split, flows within each split are
optionally sorted by a timestamp column (when available), then a sliding
window of length **T = 20** is applied.  Each window sample predicts the
label of its **last flow** (per-flow labeling).

| Dataset    | Timestamp ordering |
|------------|--------------------|
| CICIDS2017 | `Timestamp` column (if present), else row order |
| NSL-KDD    | Row order (no true timestamp in raw format) |
| UAVIDS     | `timestamp` column (if present), else row order |

### Spatial axis (configurable via `GRAPH_MODE`)
Within each window of 20 flows the within-window graph is built according to `GRAPH_MODE`:

| `GRAPH_MODE` | Edges per window | Description |
|---|---|---|
| `"fully_connected"` *(default)* | T×(T−1) + T self-loops | Every flow connects to every other flow — complete within-window spatial mixing |
| `"knn"` | sparse | Each flow connects to its k = 5 nearest neighbours in feature space |

### Model architecture
```
Input (T=20 flows, F features)
  │
  ▼
GCNConv × num_gcn_layers  ← spatial message passing over within-window graph
                             (fully-connected by default; see GRAPH_MODE)
  │
  ▼ reshape (B, T, H)
BiGRU                     ← temporal modeling over ordered window
  │ last hidden state (B, 2×gru_hidden)
  ▼
MLP classifier            ← predict label of last flow
```

### Configuration knobs
- `WINDOW_SIZE` — context-window length (default 20)
- `GRAPH_MODE` — `"fully_connected"` (default) or `"knn"` — spatial graph type within each window
- `K_SPATIAL` — kNN neighbours within each window when `GRAPH_MODE = "knn"` (default 5)
- `USE_TIMESTAMP_ORDERING` — sort by timestamp when available (default True)


In [ ]:
# ── Pipeline configuration ────────────────────────────────────────────────
# How many datasets to evaluate: 1, 2, or 'all'
NUM_DATASETS_TO_TEST = 'all'
DATASET_ORDER = ['CICIDS2017', 'NSL-KDD', 'UAVIDS']

# ── Context-window / GCN-BiGRU configuration ──────────────────────────────
# Window length T: number of consecutive flows per context window
WINDOW_SIZE = 20

# kNN neighbours for the within-window spatial graph (clipped to WINDOW_SIZE-1)
K_SPATIAL = 5

# Spatial graph mode for within-window edges.
# "fully_connected" — every flow in the window connects to every other flow (recommended).
# "knn"             — each flow connects to its K_SPATIAL nearest neighbours in feature space.
GRAPH_MODE = "fully_connected"

# Sort flows by timestamp when a timestamp column is available.
# Falls back to row order for datasets without timestamps (e.g. NSL-KDD).
USE_TIMESTAMP_ORDERING = True

# Known timestamp column names per dataset (first match wins)
_TIMESTAMP_CANDIDATES = {
    'CICIDS2017': [' Timestamp', 'Timestamp', 'Flow Start Time'],
    'NSL-KDD':    [],
    'UAVIDS':     ['timestamp', 'Timestamp', 'time', 'Time'],
}

GCN_BIGRU_PARAMS = {
    'hidden_dim':     128,
    'gru_hidden':     64,
    'num_gcn_layers': 2,
    'num_gru_layers': 1,
    'dropout':        0.2,
    'lr':             0.001,
    'weight_decay':   1e-5,
    'epochs':         50,
    'patience':       20,
    'min_delta':      1e-4,
    'grad_clip':      1.0,
    'batch_size':     256,
    'window_size':    WINDOW_SIZE,
    'k_spatial':      K_SPATIAL,
    'graph_mode':     GRAPH_MODE,
}

EXPERIMENT_CFG = {
    'new_dataset_size': 100000,
    'random_state': 42,
}


In [ ]:
def _sort_by_timestamp(X_raw: pd.DataFrame, y: pd.Series, ts_col: str):
    """Sort X_raw and y by ts_col; handles datetime strings robustly.

    Returns (X_raw_sorted, y_sorted) both with reset integer index.
    Falls back to original order if sorting fails.
    """
    try:
        col = pd.to_datetime(X_raw[ts_col], infer_datetime_format=True, errors='coerce')
        if col.isna().all():
            col = X_raw[ts_col]  # fall back to raw values
        sort_idx = col.argsort(na_position='last').values
    except Exception:
        return X_raw.reset_index(drop=True), y.reset_index(drop=True)
    return (
        X_raw.iloc[sort_idx].reset_index(drop=True),
        y.iloc[sort_idx].reset_index(drop=True),
    )


def _select_dataset_names(order: list, how_many) -> list:
    """Return a subset of dataset names based on the NUM_DATASETS_TO_TEST setting."""
    if str(how_many).lower() == 'all':
        return order[:]
    n = int(how_many)
    if n not in (1, 2, len(order)):
        raise ValueError(f"NUM_DATASETS_TO_TEST must be 1, 2, or 'all'. Got: {how_many}")
    return order[:n]


def _create_imbalanced_subset(
    df: pd.DataFrame,
    target_col: str,
    new_dataset_size: int = 5000,
    random_state: int = 42,
) -> pd.DataFrame:
    """Downsample large classes while keeping small classes intact."""
    class_counts = df[target_col].value_counts()
    large_classes = class_counts[class_counts > 500]
    small_classes = class_counts[class_counts <= 500]

    total_large = max(1, large_classes.sum())
    scaled_counts = (large_classes / total_large * new_dataset_size).astype(int)

    sampled_data = []
    for class_label, original_count in large_classes.items():
        n_sample = max(1, min(int(scaled_counts[class_label]), int(original_count)))
        sampled_data.append(
            df[df[target_col] == class_label].sample(n=n_sample, random_state=random_state)
        )
    for class_label in small_classes.index:
        sampled_data.append(df[df[target_col] == class_label])

    return pd.concat(sampled_data).reset_index(drop=True)


def _prepare_dataset_for_experiment(
    dataset_name: str,
    model_hparams: dict,
    new_dataset_size: int = 5000,
    random_state: int = 42,
) -> dict:
    """Preprocess a dataset and build train/val/test window datasets.

    * preprocess_data cleans features and target together (no alignment drift).
    * ColumnTransformer is fitted on the training split only (no data leakage).
    * Within each split, rows are optionally sorted by timestamp (when available)
      before constructing context windows — no cross-split information is used.
    * FlowWindowDataset wraps each sorted split into sliding windows of
      length WINDOW_SIZE; labels are the last-flow label per window.
    """
    target_col = target_columns[dataset_name]
    df = preprocess_data(datasets[dataset_name].copy())

    if dataset_name == 'NSL-KDD':
        df[target_col] = df[target_col].apply(_map_nsl_attack_group)

    df = df.dropna(subset=[target_col]).reset_index(drop=True)

    le = LabelEncoder()
    df[target_col] = le.fit_transform(df[target_col].astype(str))
    class_names = le.classes_

    df = _create_imbalanced_subset(
        df, target_col=target_col,
        new_dataset_size=new_dataset_size,
        random_state=random_state,
    )

    X_raw = df.drop(columns=[target_col])
    y = df[target_col]

    # 60 / 20 / 20 stratified split; fall back to non-stratified if a class is too small
    try:
        train_X_raw, tmp_X_raw, train_y, tmp_y = train_test_split(
            X_raw, y, test_size=0.4, random_state=random_state, stratify=y
        )
        val_X_raw, test_X_raw, val_y, test_y = train_test_split(
            tmp_X_raw, tmp_y, test_size=0.5, random_state=random_state, stratify=tmp_y
        )
    except ValueError:
        train_X_raw, tmp_X_raw, train_y, tmp_y = train_test_split(
            X_raw, y, test_size=0.4, random_state=random_state
        )
        val_X_raw, test_X_raw, val_y, test_y = train_test_split(
            tmp_X_raw, tmp_y, test_size=0.5, random_state=random_state
        )

    # --- Timestamp-aware ordering (within each split, no leakage) -----------
    if USE_TIMESTAMP_ORDERING:
        candidates = _TIMESTAMP_CANDIDATES.get(dataset_name, [])
        ts_col = next((c for c in candidates if c in train_X_raw.columns), None)
        if ts_col:
            print(f'  [{dataset_name}] Sorting splits by timestamp column: {repr(ts_col)}')
            train_X_raw, train_y = _sort_by_timestamp(train_X_raw, train_y, ts_col)
            val_X_raw,   val_y   = _sort_by_timestamp(val_X_raw,   val_y,   ts_col)
            test_X_raw,  test_y  = _sort_by_timestamp(test_X_raw,  test_y,  ts_col)
        else:
            print(f'  [{dataset_name}] No timestamp column found — using row order.')
            train_X_raw = train_X_raw.reset_index(drop=True)
            val_X_raw   = val_X_raw.reset_index(drop=True)
            test_X_raw  = test_X_raw.reset_index(drop=True)
            train_y     = train_y.reset_index(drop=True)
            val_y       = val_y.reset_index(drop=True)
            test_y      = test_y.reset_index(drop=True)
    else:
        train_X_raw = train_X_raw.reset_index(drop=True)
        val_X_raw   = val_X_raw.reset_index(drop=True)
        test_X_raw  = test_X_raw.reset_index(drop=True)
        train_y     = train_y.reset_index(drop=True)
        val_y       = val_y.reset_index(drop=True)
        test_y      = test_y.reset_index(drop=True)

    # Fit preprocessing on training set only (no leakage)
    preprocessor = build_preprocessor(train_X_raw)
    train_X = pd.DataFrame(preprocessor.fit_transform(train_X_raw)).reset_index(drop=True)
    val_X   = pd.DataFrame(preprocessor.transform(val_X_raw)).reset_index(drop=True)
    test_X  = pd.DataFrame(preprocessor.transform(test_X_raw)).reset_index(drop=True)
    train_y = train_y.reset_index(drop=True)
    val_y   = val_y.reset_index(drop=True)
    test_y  = test_y.reset_index(drop=True)

    # Mutual-information feature selection (fitted on train only)
    keep_ratio = 1.0
    if keep_ratio >= 1.0:
        train_X1, val_X1, test_X1 = train_X, val_X, test_X
    else:
        mi = mutual_info_classif(train_X, train_y)
        top_features = np.argsort(mi)[-max(1, int(len(mi) * keep_ratio)):]
        train_X1 = train_X.iloc[:, top_features]
        val_X1   = val_X.iloc[:, top_features]
        test_X1  = test_X.iloc[:, top_features]

    # --- Build window datasets (within each split only) -------------------
    k_spatial   = int(model_hparams.get('k_spatial', K_SPATIAL))
    graph_mode  = model_hparams.get('graph_mode', GRAPH_MODE)
    window_size = int(model_hparams.get('window_size', WINDOW_SIZE))

    train_dataset = FlowWindowDataset(
        train_X1.values, train_y.values,
        window_size=window_size, k_spatial=k_spatial, graph_mode=graph_mode,
    )
    val_dataset = FlowWindowDataset(
        val_X1.values, val_y.values,
        window_size=window_size, k_spatial=k_spatial, graph_mode=graph_mode,
    )
    test_dataset = FlowWindowDataset(
        test_X1.values, test_y.values,
        window_size=window_size, k_spatial=k_spatial, graph_mode=graph_mode,
    )

    print(
        f'  Window datasets — train: {len(train_dataset)}, '
        f'val: {len(val_dataset)}, test: {len(test_dataset)} windows'
    )

    return {
        'train_dataset': train_dataset,
        'val_dataset':   val_dataset,
        'test_dataset':  test_dataset,
        'train_y':       train_y,
        'val_y':         val_y,
        'test_y':        test_y,
        'class_names':   class_names,
    }


In [ ]:
def run_single_dataset_experiment(
    dataset_name: str,
    model_hparams: dict,
    new_dataset_size: int = 5000,
    random_state: int = 42,
) -> dict:
    """Run a full train/val/test experiment for one dataset and return results."""
    set_seed(random_state)
    bundle = _prepare_dataset_for_experiment(
        dataset_name=dataset_name,
        model_hparams=model_hparams,
        new_dataset_size=new_dataset_size,
        random_state=random_state,
    )

    num_classes = int(len(np.unique(bundle['train_y'])))
    class_weights = None
    if num_classes > 2:
        class_weights = compute_class_weights(bundle['train_y'], num_classes)
        if dataset_name == 'CICIDS2017':
            dist = bundle['train_y'].value_counts().sort_index()
            print(f'  CICIDS2017 class distribution (train split): {dict(dist)}')
            print(
                f'  Class weights — min: {class_weights.min():.4f}, '
                f'max: {class_weights.max():.4f}'
            )

    results_df, confusion_matrices, split_metrics_tables = run_all_benchmarks(
        train_dataset=bundle['train_dataset'],
        val_dataset=bundle['val_dataset'],
        test_dataset=bundle['test_dataset'],
        num_classes=num_classes,
        class_weights=class_weights,
        model_hparams=model_hparams,
    )

    model_row = results_df.loc[results_df['Model'] == 'GCN-BiGRU'].iloc[0]
    test_macro_f1 = float(model_row['F1'])
    test_accuracy = float(model_row['Accuracy'])

    payload = {
        'confusion_matrices':   confusion_matrices,
        'class_names':          bundle['class_names'],
        'split_metrics_tables': split_metrics_tables,
    }

    del bundle
    torch.cuda.empty_cache()
    gc.collect()

    return {
        'results_df':    results_df,
        'payload':       payload,
        'test_macro_f1': test_macro_f1,
        'test_accuracy': test_accuracy,
    }


In [ ]:
selected_datasets = _select_dataset_names(DATASET_ORDER, NUM_DATASETS_TO_TEST)
print(f'Selected datasets: {selected_datasets}')

dataset_results_rows = []

for idx, dataset_name in enumerate(selected_datasets, start=1):
    print(f"\n{'='*95}")
    print(f'DATASET {idx}/{len(selected_datasets)}: {dataset_name}')
    print(f"{'='*95}")

    print('\n[Step A] GCN-BiGRU experiment (context-window T=20)')
    baseline_out = run_single_dataset_experiment(
        dataset_name=dataset_name,
        model_hparams=GCN_BIGRU_PARAMS,
        new_dataset_size=EXPERIMENT_CFG['new_dataset_size'],
        random_state=EXPERIMENT_CFG['random_state'],
    )
    display(baseline_out['results_df'])

    print(
        f'[Result] {dataset_name} | '
        f"Test Macro-F1: {baseline_out['test_macro_f1']:.4f}, "
        f"Test Accuracy: {baseline_out['test_accuracy']:.4f}"
    )

    dataset_results_rows.append({
        'Dataset':        dataset_name,
        'Params':         str(GCN_BIGRU_PARAMS),
        'Test Accuracy':  baseline_out['test_accuracy'],
        'Test Macro-F1':  baseline_out['test_macro_f1'],
    })

BASELINE_RESULTS_DF = pd.DataFrame(dataset_results_rows)
print('\nFinal summary: GCN-BiGRU results')
display(BASELINE_RESULTS_DF)


## Visualization

In [ ]:
# Baseline Macro-F1 per dataset
if 'BASELINE_RESULTS_DF' not in globals() or BASELINE_RESULTS_DF.empty:
    print('Run the pipeline cell first to generate BASELINE_RESULTS_DF.')
else:
    plot_df = BASELINE_RESULTS_DF.copy()
    x = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=(max(7, len(plot_df) * 2.2), 4.2))
    bars = ax.bar(x, plot_df['Test Macro-F1'], width=0.5, label='Baseline (Default)', color='#9E9E9E')

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, min(1.03, h + 0.01),
                f'{h:.3f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['Dataset'].tolist())
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Test Macro-F1')
    ax.set_xlabel('Dataset')
    ax.set_title('Baseline BG-GCN: Test Macro-F1 per Dataset')
    ax.grid(axis='y', alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Baseline Accuracy per dataset
if 'BASELINE_RESULTS_DF' not in globals() or BASELINE_RESULTS_DF.empty:
    print('Run the pipeline cell first to generate BASELINE_RESULTS_DF.')
else:
    plot_df = BASELINE_RESULTS_DF.copy()
    x = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=(max(7, len(plot_df) * 2.2), 4.2))
    bars = ax.bar(x, plot_df['Test Accuracy'], width=0.5, label='Baseline (Default)', color='#1565C0')

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, min(1.03, h + 0.01),
                f'{h:.3f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['Dataset'].tolist())
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Test Accuracy')
    ax.set_xlabel('Dataset')
    ax.set_title('Baseline BG-GCN: Test Accuracy per Dataset')
    ax.grid(axis='y', alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()
